# F5.news News Topics

Find topics getting coverage, with related stories and article links under each topic. Counts measure coverage in one GDELT batch, not growing interest or reader sentiment.

Install `requirements-semantic.txt`, or run `./resources/data-analysis/run.sh --gdelt` to create the environment and open the report. No MongoDB credentials are needed.


In [ ]:
from pathlib import Path
import json
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "resources/data-analysis").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the F5.news repository.")
ANALYSIS_DIR = ROOT / "resources/data-analysis"
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

from run_gdelt import pull_latest_batch, filter_country
from gdelt_news_filter import filter_news
from gdelt_topics import build_topics
from gdelt_topic_report import render_topics
from IPython.display import HTML, display

USE_SAVED_BATCH = True  # False downloads the latest batch.
INCLUDE_ALL = False  # True also keeps quizzes, advice, and entertainment.
OUTPUT_DIR = ANALYSIS_DIR / "models/gdelt"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Read News

Start with the saved US sample. Set `USE_SAVED_BATCH = False` above for a fresh download. The US-source filter uses an old publisher directory and can misclassify outlets.


In [ ]:
sample_path = OUTPUT_DIR / "latest-us.json"
if USE_SAVED_BATCH:
    if not sample_path.exists():
        raise RuntimeError("No saved batch. Set USE_SAVED_BATCH = False and rerun.")
    news = json.loads(sample_path.read_text())
else:
    news = filter_country(pull_latest_batch(), "US")
    sample_path.write_text(json.dumps(news, indent=2), encoding="utf-8")
print(f"Read {len(news['articles']):,} articles.")


## Find Topics

Headlines are compared by meaning on this computer. Related stories about the same person share one topic, such as Christa Pike. Other topics use headline phrases. Unmatched articles stay under Other News.


In [ ]:
filtered_news = news if INCLUDE_ALL else filter_news(news)
news_topics = build_topics(filtered_news)
report_path = OUTPUT_DIR / "latest-us-topics.html"
(OUTPUT_DIR / "latest-us-topics.json").write_text(json.dumps(news_topics, indent=2), encoding="utf-8")
report_html = render_topics(news_topics)
report_path.write_text(report_html, encoding="utf-8")
print(f"{len(news_topics['topics']):,} groups from {news_topics['article_count']:,} news articles.")
print(f"Report: {report_path}")
display(HTML(report_html))
